# Model Training

- Goal: train baseline models and evaluate them using fraud-focused metrics.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)


## 1. Load Prepared Data

- Load the deduplicated dataset that will be used for training.

In [2]:
data_path = "../dataset/creditcard.csv"
df = pd.read_csv(data_path)
df = df.drop_duplicates()

df.shape

(283726, 31)

## 2. Train / Validation / Test Split

- Split the dataset into stratified train, validation, and test sets using the chosen `75 / 15 / 10` strategy.

In [3]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, 
    y, 
    test_size=0.10, 
    stratify=y, 
    random_state=42
    )

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, 
    y_temp, 
    test_size=15/90, 
    stratify=y_temp, 
    random_state=42
    )


In [4]:
print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)


Train shape: (212794, 30) (212794,)
Validation shape: (42559, 30) (42559,)
Test shape: (28373, 30) (28373,)


In [5]:
print("Overall class distribution:")
print(y.value_counts(normalize=True))

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation class distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))


Overall class distribution:
Class
0    0.998333
1    0.001667
Name: proportion, dtype: float64

Train class distribution:
Class
0    0.998332
1    0.001668
Name: proportion, dtype: float64

Validation class distribution:
Class
0    0.998332
1    0.001668
Name: proportion, dtype: float64

Test class distribution:
Class
0    0.998343
1    0.001657
Name: proportion, dtype: float64


## 3. Dummy Baseline

- Create a baseline that predicts every transaction as non-fraud.

In [6]:
# Predict every transaction as non-fraud
y_pred_dummy = np.zeros_like(y_val)

# Fraud scores for metrics like PR-AUC and ROC-AUC
y_score_dummy = np.zeros_like(y_val, dtype=float)

## 4. Baseline Evaluation

- Measure how the dummy baseline performs on the validation set.

In [7]:
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_dummy))

print("\nPrecision:", precision_score(y_val, y_pred_dummy, zero_division=0))
print("Recall:", recall_score(y_val, y_pred_dummy, zero_division=0))
print("F1 Score:", f1_score(y_val, y_pred_dummy, zero_division=0))
print("PR-AUC:", average_precision_score(y_val, y_score_dummy))
print("ROC-AUC:", roc_auc_score(y_val, y_score_dummy))

Confusion Matrix:
[[42488     0]
 [   71     0]]

Precision: 0.0
Recall: 0.0
F1 Score: 0.0
PR-AUC: 0.0016682722808336662
ROC-AUC: 0.5


## 5. Logistic Regression Baseline

- Train the first real baseline model using logistic regression.

In [8]:
log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=1000))
])

log_reg_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

## 6. Logistic Regression Evaluation

- Evaluate logistic regression using `precision`, `recall`, `F1`, `PR-AUC`, and `ROC-AUC`.

In [9]:
y_pred_logreg = log_reg_model.predict(X_val)
y_score_logreg = log_reg_model.predict_proba(X_val)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_logreg))

print("\nPrecision:", precision_score(y_val, y_pred_logreg, zero_division=0))
print("Recall:", recall_score(y_val, y_pred_logreg, zero_division=0))
print("F1 Score:", f1_score(y_val, y_pred_logreg, zero_division=0))
print("PR-AUC:", average_precision_score(y_val, y_score_logreg))
print("ROC-AUC:", roc_auc_score(y_val, y_score_logreg))


Confusion Matrix:
[[42480     8]
 [   27    44]]

Precision: 0.8461538461538461
Recall: 0.6197183098591549
F1 Score: 0.7154471544715447
PR-AUC: 0.7228271628719523
ROC-AUC: 0.973917407665727


## 7. Baseline Comparison

- Compare the dummy baseline and logistic regression to see whether the trained model is meaningfully better.

In [10]:
comparison = {
    "Dummy Baseline": {
        "Precision": precision_score(y_val, y_pred_dummy, zero_division=0),
        "Recall": recall_score(y_val, y_pred_dummy, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_dummy, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_dummy),
        "ROC-AUC": roc_auc_score(y_val, y_score_dummy)
    },
    "Logistic Regression": {
        "Precision": precision_score(y_val, y_pred_logreg, zero_division=0),
        "Recall": recall_score(y_val, y_pred_logreg, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_logreg, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_logreg),
        "ROC-AUC": roc_auc_score(y_val, y_score_logreg)
    }
}

comparison_df = pd.DataFrame(comparison).T
comparison_df


,Precision,Recall,F1 Score,PR-AUC,ROC-AUC
Dummy Baseline,0.000000,0.000000,0.000000,0.001668,0.500000
Logistic Regression,0.846154,0.619718,0.715447,0.722827,0.973917


## Observations

- The logistic regression baseline performs far better than the dummy baseline on every meaningful fraud metric.
- `PR-AUC = 0.7228` is a strong starting point and is the most important result since `PR-AUC` is the primary evaluation metric.
- `Recall = 0.6197` means the model is catching about 62% of fraud cases on the validation set.
- `Precision = 0.8462` is high, which means the model is not flagging many non-fraud transactions as fraud.
- `F1 Score = 0.7154` suggests that the balance between precision and recall is already reasonably strong.
- `ROC-AUC = 0.9739` is very high, but `PR-AUC` is still more important for interpreting performance on this imbalanced dataset.
- The dummy baseline confirms that predicting all transactions as non-fraud is useless for fraud detection even if it seems safe.
- Logistic regression is a meaningful baseline and gives a solid reference point before trying more advanced models.
- The next improvement should focus on threshold tuning and the precision-recall tradeoff rather than immediately switching to a more complex model.

## 8. Basic Tree Model

- Train and evaluate a simple tree-based model as another baseline for comparison.

In [11]:
tree_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,
    class_weight="balanced"
)

tree_model.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42)

In [12]:
y_pred_tree = tree_model.predict(X_val)
y_score_tree = tree_model.predict_proba(X_val)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_tree))

print("\nPrecision:", precision_score(y_val, y_pred_tree, zero_division=0))
print("Recall:", recall_score(y_val, y_pred_tree, zero_division=0))
print("F1 Score:", f1_score(y_val, y_pred_tree, zero_division=0))
print("PR-AUC:", average_precision_score(y_val, y_score_tree))
print("ROC-AUC:", roc_auc_score(y_val, y_score_tree))


Confusion Matrix:
[[41036  1452]
 [    9    62]]

Precision: 0.04095112285336856
Recall: 0.8732394366197183
F1 Score: 0.07823343848580441
PR-AUC: 0.4250353798847824
ROC-AUC: 0.9368264046716752


In [13]:
comparison = {
    "Dummy Baseline": {
        "Precision": precision_score(y_val, y_pred_dummy, zero_division=0),
        "Recall": recall_score(y_val, y_pred_dummy, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_dummy, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_dummy),
        "ROC-AUC": roc_auc_score(y_val, y_score_dummy)
    },
    "Logistic Regression": {
        "Precision": precision_score(y_val, y_pred_logreg, zero_division=0),
        "Recall": recall_score(y_val, y_pred_logreg, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_logreg, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_logreg),
        "ROC-AUC": roc_auc_score(y_val, y_score_logreg)
    },
    "Decision Tree": {
        "Precision": precision_score(y_val, y_pred_tree, zero_division=0),
        "Recall": recall_score(y_val, y_pred_tree, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_tree, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_tree),
        "ROC-AUC": roc_auc_score(y_val, y_score_tree)
    }
}

comparison_df = pd.DataFrame(comparison).T
comparison_df


,Precision,Recall,F1 Score,PR-AUC,ROC-AUC
Dummy Baseline,0.000000,0.000000,0.000000,0.001668,0.500000
Logistic Regression,0.846154,0.619718,0.715447,0.722827,0.973917
Decision Tree,0.040951,0.873239,0.078233,0.425035,0.936826


## Tree Model Observations

- The decision tree catches more frauds than logistic regression, with recall increasing to about 87%.
- Precision is extremely low at about 4.1%, which means the tree is flagging a very large number of legitimate transactions as fraud.
- `F1 Score = 0.0782` is far below logistic regression, so the gain in recall is coming at too large a cost.
- `PR-AUC = 0.4250` is much worse than logistic regression, so the tree is weaker by the primary evaluation metric.
- `ROC-AUC = 0.9368` is still high, but this is another reminder that ROC-AUC can look strong even when the model is not practically useful enough.
- The decision tree behaves like an overly aggressive fraud detector.
- Logistic regression remains the better baseline overall because it keeps high precision and a much stronger precision-recall tradeoff.

## 9. Next Training Steps

- Decide whether to tune thresholds, adjust class imbalance handling, or move to a more complex model.

Proceed with threshold tuning first. Why threshold tuning next?

- Logistic model already has strong ranking performance.
- PR-AUC is good, which means the probabilities are useful.
- Right now we are only using the default threshold of 0.5.
- Fraud problems usually get a lot of benefit from changing the threshold to trade precision for recall deliberately.

In [14]:
thresholds = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7]
y_score_logreg = log_reg_model.predict_proba(X_val)[:, 1]
for threshold in thresholds:
    y_pred_thresh_logreg = (y_score_logreg >= threshold).astype(int)
    print(f"Threshold: {threshold}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred_thresh_logreg))
    print("\nPrecision:", precision_score(y_val, y_pred_thresh_logreg, zero_division=0))
    print("Recall:", recall_score(y_val, y_pred_thresh_logreg, zero_division=0))
    print("F1 Score:", f1_score(y_val, y_pred_thresh_logreg, zero_division=0))
    print("-" * 30)

    
    

Threshold: 0.01
Confusion Matrix:
[[42391    97]
 [   12    59]]

Precision: 0.3782051282051282
Recall: 0.8309859154929577
F1 Score: 0.5198237885462555
------------------------------
Threshold: 0.05
Confusion Matrix:
[[42462    26]
 [   15    56]]

Precision: 0.6829268292682927
Recall: 0.7887323943661971
F1 Score: 0.7320261437908496
------------------------------
Threshold: 0.1
Confusion Matrix:
[[42470    18]
 [   17    54]]

Precision: 0.75
Recall: 0.7605633802816901
F1 Score: 0.7552447552447552
------------------------------
Threshold: 0.3
Confusion Matrix:
[[42473    15]
 [   23    48]]

Precision: 0.7619047619047619
Recall: 0.676056338028169
F1 Score: 0.7164179104477612
------------------------------
Threshold: 0.5
Confusion Matrix:
[[42480     8]
 [   27    44]]

Precision: 0.8461538461538461
Recall: 0.6197183098591549
F1 Score: 0.7154471544715447
------------------------------
Threshold: 0.7
Confusion Matrix:
[[42482     6]
 [   32    39]]

Precision: 0.8666666666666667
Recall: 

### Threshold Tuning Observations

- `0.1` is the best middle-ground threshold among the tested values.
- It gives the highest `F1 Score`, which means it provides the strongest balance between precision and recall on the validation set.
- At `0.1`, precision and recall are also very close to each other, so the model is not leaning too heavily toward only one side of the tradeoff.
- `0.01` and `0.05` are more recall-focused choices.
- These lower thresholds catch more fraud, but they also increase false positives and reduce precision.
- `0.3`, `0.5`, and `0.7` are more precision-focused choices.
- These higher thresholds reduce false positives, but they also miss more fraud cases because recall drops.
- If the goal is balanced fraud detection performance, `0.1` is the strongest candidate.
- If the goal is to catch as much fraud as possible, a lower threshold such as `0.05` may be preferred.
- If the goal is to reduce false alarms as much as possible, a higher threshold such as `0.5` or `0.7` may be preferred.

## Class Balanced Logistic Regression

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

bal_logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
])

bal_logreg_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [16]:
y_bal_pred_logreg = bal_logreg_model.predict(X_val)
y_bal_score_logreg = bal_logreg_model.predict_proba(X_val)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_bal_pred_logreg))
print("\nPrecision:", precision_score(y_val, y_bal_pred_logreg, zero_division=0))
print("Recall:", recall_score(y_val, y_bal_pred_logreg, zero_division=0))
print("F1 Score:", f1_score(y_val, y_bal_pred_logreg, zero_division=0))
print("PR-AUC:", average_precision_score(y_val, y_bal_score_logreg))
print("ROC-AUC:", roc_auc_score(y_val, y_bal_score_logreg)) 

Confusion Matrix:
[[41576   912]
 [    7    64]]

Precision: 0.06557377049180328
Recall: 0.9014084507042254
F1 Score: 0.12225405921680993
PR-AUC: 0.6997720028289108
ROC-AUC: 0.9724576748762203


In [17]:
thresholds = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7]
y_bal_score_logreg = bal_logreg_model.predict_proba(X_val)[:, 1]
for threshold in thresholds:
    y_pred_thresh_logreg = (y_bal_score_logreg >= threshold).astype(int)
    print(f"Threshold: {threshold}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred_thresh_logreg))
    print("\nPrecision:", precision_score(y_val, y_pred_thresh_logreg, zero_division=0))
    print("Recall:", recall_score(y_val, y_pred_thresh_logreg, zero_division=0))
    print("F1 Score:", f1_score(y_val, y_pred_thresh_logreg, zero_division=0))
    print("-" * 30)

Threshold: 0.01
Confusion Matrix:
[[11194 31294]
 [    0    71]]

Precision: 0.002263669695520485
Recall: 1.0
F1 Score: 0.004517114136658608
------------------------------
Threshold: 0.05
Confusion Matrix:
[[28153 14335]
 [    2    69]]

Precision: 0.0047903360177728406
Recall: 0.971830985915493
F1 Score: 0.009533678756476684
------------------------------
Threshold: 0.1
Confusion Matrix:
[[33255  9233]
 [    4    67]]

Precision: 0.007204301075268817
Recall: 0.9436619718309859
F1 Score: 0.014299434425354819
------------------------------
Threshold: 0.3
Confusion Matrix:
[[40072  2416]
 [    5    66]]

Precision: 0.0265914585012087
Recall: 0.9295774647887324
F1 Score: 0.05170387779083431
------------------------------
Threshold: 0.5
Confusion Matrix:
[[41576   912]
 [    7    64]]

Precision: 0.06557377049180328
Recall: 0.9014084507042254
F1 Score: 0.12225405921680993
------------------------------
Threshold: 0.7
Confusion Matrix:
[[42049   439]
 [    8    63]]

Precision: 0.1254980079

In [18]:
thresholds = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7]

comparison_rows = []

for threshold in thresholds:
    y_pred_unweighted = (y_score_logreg >= threshold).astype(int)
    y_pred_weighted = (y_bal_score_logreg >= threshold).astype(int)

    comparison_rows.append({
        "Threshold": threshold,

        "Unweighted Precision": precision_score(y_val, y_pred_unweighted, zero_division=0),
        "Weighted Precision": precision_score(y_val, y_pred_weighted, zero_division=0),
        "Unweighted Recall": recall_score(y_val, y_pred_unweighted, zero_division=0),
        "Weighted Recall": recall_score(y_val, y_pred_weighted, zero_division=0),
        "Unweighted F1": f1_score(y_val, y_pred_unweighted, zero_division=0),
        "Weighted F1": f1_score(y_val, y_pred_weighted, zero_division=0),

    })

threshold_comparison_df = pd.DataFrame(comparison_rows)

threshold_comparison_df

,Threshold,Unweighted Precision,Weighted Precision,Unweighted Recall,Weighted Recall,Unweighted F1,Weighted F1
0,0.01,0.378205,0.002264,0.830986,1.000000,0.519824,0.004517
1,0.05,0.682927,0.004790,0.788732,0.971831,0.732026,0.009534
2,0.10,0.750000,0.007204,0.760563,0.943662,0.755245,0.014299
3,0.30,0.761905,0.026591,0.676056,0.929577,0.716418,0.051704
4,0.50,0.846154,0.065574,0.619718,0.901408,0.715447,0.122254
5,0.70,0.866667,0.125498,0.549296,0.887324,0.672414,0.219895


## Weighted vs Unweighted Logistic Regression Insights

- The unweighted logistic regression performs better than the weighted version at every tested threshold when judged by `F1 Score`.
- The weighted logistic regression consistently achieves much higher recall, but this comes at an extreme cost to precision.
- Even at higher thresholds such as `0.5` and `0.7`, the weighted model still has very weak precision compared to the unweighted model.
- This means the weighted model is aggressively flagging legitimate transactions as fraud.
- The unweighted model provides a much stronger precision-recall balance across the full threshold range.
- The best overall middle-ground result still comes from the unweighted model at threshold `0.1`, where it achieves the highest `F1 Score`.
- These results suggest that `class_weight="balanced"` is not improving performance for this dataset and this model setup.
- For this project, the stronger direction is to continue with unweighted logistic regression and use threshold tuning to control the fraud-detection tradeoff.

## Export Useful Training Outputs

- Save the main training comparison tables and a short summary into `outputs/train/`.

In [19]:
from pathlib import Path

output_dir = Path("../outputs/train")
output_dir.mkdir(parents=True, exist_ok=True)

# Rebuild the most useful threshold tables so they are easy to save.
thresholds = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7]

logreg_threshold_rows = []
for threshold in thresholds:
    y_pred_thresh = (y_score_logreg >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh).ravel()
    logreg_threshold_rows.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh, zero_division=0)
    })

logreg_threshold_df = pd.DataFrame(logreg_threshold_rows)
logreg_threshold_df.to_csv(output_dir / "logreg_threshold_results.csv", index=False)

weighted_logreg_threshold_rows = []
for threshold in thresholds:
    y_pred_thresh = (y_bal_score_logreg >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_thresh).ravel()
    weighted_logreg_threshold_rows.append({
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_val, y_pred_thresh, zero_division=0),
        "Recall": recall_score(y_val, y_pred_thresh, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_thresh, zero_division=0)
    })

weighted_logreg_threshold_df = pd.DataFrame(weighted_logreg_threshold_rows)
weighted_logreg_threshold_df.to_csv(output_dir / "weighted_logreg_threshold_results.csv", index=False)

logreg_vs_weighted_df = threshold_comparison_df.copy()
logreg_vs_weighted_df.to_csv(output_dir / "logreg_vs_weighted_logreg_comparison.csv", index=False)

baseline_models_df = pd.DataFrame({
    "Dummy Baseline": {
        "Precision": precision_score(y_val, y_pred_dummy, zero_division=0),
        "Recall": recall_score(y_val, y_pred_dummy, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_dummy, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_dummy),
        "ROC-AUC": roc_auc_score(y_val, y_score_dummy)
    },
    "Logistic Regression": {
        "Precision": precision_score(y_val, y_pred_logreg, zero_division=0),
        "Recall": recall_score(y_val, y_pred_logreg, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_logreg, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_logreg),
        "ROC-AUC": roc_auc_score(y_val, y_score_logreg)
    },
    "Decision Tree": {
        "Precision": precision_score(y_val, y_pred_tree, zero_division=0),
        "Recall": recall_score(y_val, y_pred_tree, zero_division=0),
        "F1 Score": f1_score(y_val, y_pred_tree, zero_division=0),
        "PR-AUC": average_precision_score(y_val, y_score_tree),
        "ROC-AUC": roc_auc_score(y_val, y_score_tree)
    }
}).T
baseline_models_df.to_csv(output_dir / "baseline_model_comparison.csv")

best_logreg_row = logreg_threshold_df.loc[logreg_threshold_df["F1 Score"].idxmax()]
best_weighted_logreg_row = weighted_logreg_threshold_df.loc[weighted_logreg_threshold_df["F1 Score"].idxmax()]

train_notes = f"""Training outputs saved from train.ipynb.

Key conclusions:
- Best unweighted logistic regression threshold by F1: {best_logreg_row['Threshold']}
- Best unweighted logistic regression F1: {best_logreg_row['F1 Score']:.4f}
- Best weighted logistic regression threshold by F1: {best_weighted_logreg_row['Threshold']}
- Best weighted logistic regression F1: {best_weighted_logreg_row['F1 Score']:.4f}
- Weighted logistic regression did not outperform unweighted logistic regression on F1.
- Unweighted logistic regression remains the stronger training-stage model direction.
"""

(output_dir / "train_notes.txt").write_text(train_notes)

print(f"Saved training outputs to: {output_dir.resolve()}")

Saved training outputs to: /Users/nidhushankanagaraja/Desktop/NYU/Personal/Learn/ML/Credit-Card-Fraud-Detection/outputs/train
